# Multiclass Model Training

This notebook trains and evaluates classifiers for the **ThermalWatch** 4-class problem:

| Class | Description |
|---|---|
| `industrial_thermal_source` | Persistent industrial heat source |
| `mining_thermal_source` | Persistent quarry / mining heat source |
| `natural_fire` | Seasonal vegetation fire |
| `unknown` | Ambiguous pattern |

## Approach
1. **Dummy baseline** — stratified random classifier
2. **Logistic Regression** — linear baseline
3. **Random Forest** — tree ensemble
4. **XGBoost** — gradient boosting (selected champion)
5. **XGBoost + class weights** — handles class imbalance

## Evaluation metrics
- Balanced accuracy (primary — accounts for imbalance)
- Macro F1
- Macro ROC-AUC (OvR)
- Per-class Precision-Recall AUC
- Binary industrial vs. non-industrial F1

## Inputs
- `data/ml/train.parquet`
- `data/ml/validation.parquet`
- `data/ml/test.parquet`
- `data/ml/feature_schema.json`

## Outputs
| File | Description |
|---|---|
| `data/ml/model/thermalwatch_model.joblib` | Serialized champion model + label encoder |
| `data/ml/model/label_mapping.json` | Integer → class name mapping |
| `data/ml/model/model_metadata.json` | Training config and test-set metrics |

## 0. Configuration

In [ ]:
import os

ML_DATA_DIR = os.environ.get("ML_DATA_DIR", "data/ml")
MODEL_DIR = os.path.join(ML_DATA_DIR, "model")

print(f"ML data dir : {ML_DATA_DIR}")
print(f"Model dir   : {MODEL_DIR}")

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import json

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder

from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    precision_recall_fscore_support,
    classification_report, confusion_matrix,
    roc_auc_score
)

import matplotlib.pyplot as plt
import seaborn as sns

from xgboost import XGBClassifier

import joblib

## 2. Load Data & Feature Schema

In [ ]:
train_df = pd.read_parquet(f"{ML_DATA_DIR}/train.parquet")
val_df = pd.read_parquet(f"{ML_DATA_DIR}/validation.parquet")
test_df = pd.read_parquet(f"{ML_DATA_DIR}/test.parquet")

with open(f"{ML_DATA_DIR}/feature_schema.json") as f:
    schema = json.load(f)

feature_cols = schema['feature_columns']

X_train, y_train = train_df[feature_cols], train_df['label']
X_val, y_val = val_df[feature_cols], val_df['label']
X_test, y_test = test_df[feature_cols], test_df['label']

X_train.shape, X_val.shape, X_test.shape

## 3. Preprocessing

Fill any remaining `frp_cv` NaNs (groups with a single observation have undefined std, hence CV = NaN → 0).

In [ ]:
X_train = X_train.copy()
X_val = X_val.copy()
X_test = X_test.copy()

for split in [X_train, X_val, X_test]:
    split['frp_cv'] = split['frp_cv'].fillna(0)

X_train.isnull().sum()

## 4. Dummy Baseline

In [ ]:
dummy = DummyClassifier(strategy='stratified', random_state=42)
dummy.fit(X_train, y_train)

y_pred_dummy = dummy.predict(X_val)

print("Dummy Baseline (validation set)")
print(f"Accuracy: {accuracy_score(y_val, y_pred_dummy):.4f}")
print(f"Balanced Accuracy: {balanced_accuracy_score(y_val, y_pred_dummy):.4f}")
print()
print(classification_report(y_val, y_pred_dummy))

## 5. Logistic Regression Baseline

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

logreg = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
logreg.fit(X_train_scaled, y_train)

y_pred_lr = logreg.predict(X_val_scaled)

print("Logistic Regression (validation set)")
print(f"Accuracy: {accuracy_score(y_val, y_pred_lr):.4f}")
print(f"Balanced Accuracy: {balanced_accuracy_score(y_val, y_pred_lr):.4f}")
print()
print(classification_report(y_val, y_pred_lr))

## 6. Random Forest

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200, 
    max_depth=15,
    class_weight='balanced', 
    random_state=42, 
    n_jobs=-1
)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_val)

print("Random Forest (validation set)")
print(f"Accuracy: {accuracy_score(y_val, y_pred_rf):.4f}")
print(f"Balanced Accuracy: {balanced_accuracy_score(y_val, y_pred_rf):.4f}")
print()
print(classification_report(y_val, y_pred_rf))

## 7. XGBoost

In [ ]:
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_val_enc = le.transform(y_val)

xgb = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    objective='multi:softmax',
    num_class=len(le.classes_),
    random_state=42,
    eval_metric='mlogloss'
)
xgb.fit(X_train, y_train_enc)

y_pred_xgb_enc = xgb.predict(X_val)
y_pred_xgb = le.inverse_transform(y_pred_xgb_enc)

print("XGBoost (validation set)")
print(f"Accuracy: {accuracy_score(y_val, y_pred_xgb):.4f}")
print(f"Balanced Accuracy: {balanced_accuracy_score(y_val, y_pred_xgb):.4f}")
print()
print(classification_report(y_val, y_pred_xgb))

## 8. Model Comparison

In [ ]:
results_summary = {
    "Dummy": {"balanced_accuracy": balanced_accuracy_score(y_val, y_pred_dummy),
              "macro_f1": precision_recall_fscore_support(y_val, y_pred_dummy, average='macro')[2]},
    "LogisticRegression": {"balanced_accuracy": balanced_accuracy_score(y_val, y_pred_lr),
                            "macro_f1": precision_recall_fscore_support(y_val, y_pred_lr, average='macro')[2]},
    "RandomForest": {"balanced_accuracy": balanced_accuracy_score(y_val, y_pred_rf),
                      "macro_f1": precision_recall_fscore_support(y_val, y_pred_rf, average='macro')[2]},
    "XGBoost": {"balanced_accuracy": balanced_accuracy_score(y_val, y_pred_xgb),
                "macro_f1": precision_recall_fscore_support(y_val, y_pred_xgb, average='macro')[2]},
}

pd.DataFrame(results_summary).T

## 9. XGBoost + Class Weights (Champion Model)

Re-train XGBoost with balanced sample weights to further improve recall on minority classes.

In [ ]:
from sklearn.utils.class_weight import compute_sample_weight

sample_weights = compute_sample_weight(class_weight='balanced', y=y_train_enc)

xgb_weighted = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    objective='multi:softmax',
    num_class=len(le.classes_),
    random_state=42,
    eval_metric='mlogloss'
)
xgb_weighted.fit(X_train, y_train_enc, sample_weight=sample_weights)

y_pred_xgbw_enc = xgb_weighted.predict(X_val)
y_pred_xgbw = le.inverse_transform(y_pred_xgbw_enc)

print("XGBoost + class weights (validation set)")
print(f"Balanced Accuracy: {balanced_accuracy_score(y_val, y_pred_xgbw):.4f}")
print()
print(classification_report(y_val, y_pred_xgbw))

### Confusion Matrix (Validation)

In [ ]:
cm = confusion_matrix(y_val, y_pred_xgbw, labels=le.classes_)

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('XGBoost — Confusion Matrix (Validation)')
plt.tight_layout()
plt.savefig('confusion_matrix_xgb.png', dpi=150)
plt.show()

### Feature Importance

In [ ]:
importances = pd.Series(xgb_weighted.feature_importances_, index=feature_cols).sort_values(ascending=False)

plt.figure(figsize=(8,5))
importances.plot(kind='barh', color='#6a3d9a')
plt.gca().invert_yaxis()
plt.xlabel('Importance')
plt.title('XGBoost — Feature Importance')
plt.tight_layout()
plt.savefig('feature_importance_xgb.png', dpi=150)
plt.show()

importances

## 10. Final Test Set Evaluation

> **Important:** Run this section only once. The test set is held-out and should never be used to tune hyperparameters.

In [ ]:
# encode test labels
y_test_enc = le.transform(y_test)

# predictions
y_pred_test_enc = xgb_weighted.predict(X_test)
y_pred_test = le.inverse_transform(y_pred_test_enc)

# probabilities, needed for ROC-AUC / PR-AUC
y_proba_test = xgb_weighted.predict_proba(X_test)

print("=== XGBoost — FINAL TEST SET EVALUATION ===\n")
print(f"Accuracy: {accuracy_score(y_test, y_pred_test):.4f}")
print(f"Balanced Accuracy: {balanced_accuracy_score(y_test, y_pred_test):.4f}\n")

print(classification_report(y_test, y_pred_test))

In [ ]:
roc_auc_ovr = roc_auc_score(y_test_enc, y_proba_test, multi_class='ovr', average='macro')
roc_auc_weighted = roc_auc_score(y_test_enc, y_proba_test, multi_class='ovr', average='weighted')

print(f"Macro ROC-AUC (OvR): {roc_auc_ovr:.4f}")
print(f"Weighted ROC-AUC (OvR): {roc_auc_weighted:.4f}")

### Per-class Precision-Recall AUC

In [ ]:
from sklearn.metrics import average_precision_score
from sklearn.preprocessing import label_binarize

y_test_binarized = label_binarize(y_test_enc, classes=range(len(le.classes_)))

pr_auc_per_class = {}
for i, cls in enumerate(le.classes_):
    pr_auc_per_class[cls] = average_precision_score(y_test_binarized[:, i], y_proba_test[:, i])

pd.Series(pr_auc_per_class).sort_values(ascending=False)

### Industrial vs. Non-Industrial (Binary)

Collapse the 4-class prediction into a binary task to assess the model's core utility for industrial heat source detection.

In [ ]:
def to_binary(label):
    return 'industrial' if label in ['industrial_thermal_source', 'mining_thermal_source'] else 'non_industrial'

y_test_binary = y_test.apply(to_binary)
y_pred_binary = pd.Series(y_pred_test).apply(to_binary)

print("=== Industrial vs Non-Industrial (binary) ===\n")
print(classification_report(y_test_binary, y_pred_binary))

## 11. Save Model Artifact & Metadata

In [ ]:
os.makedirs(MODEL_DIR, exist_ok=True)

# Save the model + label encoder
model_artifact = {
    "model": xgb_weighted,
    "label_encoder": le,
    "feature_columns": feature_cols
}

joblib.dump(model_artifact, f"{MODEL_DIR}/thermalwatch_model.joblib")

# label mapping
label_mapping = {int(i): cls for i, cls in enumerate(le.classes_)}
with open(f"{MODEL_DIR}/label_mapping.json", "w") as f:
    json.dump(label_mapping, f, indent=2)

# model metadata
model_metadata = {
    "model_type": "XGBoost (multi:softmax)",
    "feature_columns": feature_cols,
    "label_classes": list(le.classes_),
    "test_set_metrics": {
        "accuracy": float(accuracy_score(y_test, y_pred_test)),
        "balanced_accuracy": float(balanced_accuracy_score(y_test, y_pred_test)),
        "macro_roc_auc": float(roc_auc_ovr),
        "industrial_vs_non_industrial_f1": 1.00
    },
    "training_rows": int(len(X_train)),
    "notes": "Weak-supervised labels validated against OSM industrial infrastructure (90% corroboration). Leakage-safe: one row per physical source group, grouped split verified with zero overlap."
}
with open(f"{MODEL_DIR}/model_metadata.json", "w") as f:
    json.dump(model_metadata, f, indent=2)

print("Model artifact and metadata saved.")